# HV-Doc-Data — Boundary-Aware U-Net Training

Run **top to bottom** on Google Colab (`Runtime → Change runtime type → T4 GPU`).

**What this notebook does:**
1. Downloads `yashvardhangera/hv-doc-data` from Kaggle via an interactive token prompt
2. Copies the dataset into `/content/` so it's visible in the file browser
3. Loads all three annotation rounds and builds a majority-vote ground-truth mask
4. Trains a Boundary-Aware U-Net with augmentations and a cosine LR schedule
5. Saves the best checkpoint, then generates `pred.csv` over the test set

**Before you start:** go to [kaggle.com/settings](https://www.kaggle.com/settings) → **API** → **Create New Token** — you'll be asked to paste your username and key in the auth cell below.

## 1 — Install dependencies

In [ ]:
!pip install -q kagglehub albumentations tqdm kornia
print("done installing")

## 2 — Kaggle authentication

Run this cell and follow the prompt to paste your Kaggle username and API key.

In [ ]:
import kagglehub

kagglehub.login()

## 3 — Download and copy dataset to `/content/`

`kagglehub` caches to `/root/.cache/...` — not visible in the Colab file browser. This cell downloads the dataset, finds the actual data directory (the Kaggle package nests it under `student/student/`), then copies images and labels into `/content/dataset/` where you can browse them.

In [ ]:
import kagglehub, os, shutil, glob

# Step 1 — download to the kagglehub cache (~/.cache/kagglehub/...)
_cache_path = kagglehub.dataset_download("yashvardhangera/hv-doc-data")
print("Kaggle cache path:", _cache_path)

# Step 2 — find the subdirectory that actually contains images/ and labels/
def _find_data_root(base):
    for root, dirs, _ in os.walk(base):
        if "images" in dirs and "labels" in dirs:
            return root
    raise RuntimeError(f"Could not find a folder with both images/ and labels/ under {base}")

_data_root = _find_data_root(_cache_path)
print("Data root found at:", _data_root)

# Step 3 — copy into /content/dataset/ so it's visible in the file browser
CONTENT_DATA = "/content/dataset"

if os.path.exists(CONTENT_DATA):
    print(f"{CONTENT_DATA} already exists, skipping copy")
else:
    print("Copying to /content/dataset/ ...")
    shutil.copytree(_data_root, CONTENT_DATA)
    print("Done.")

# Step 4 — copy seg_common.py to /content/ so it's importable
_sc_src = os.path.join(_cache_path, "seg_common.py")
if os.path.exists(_sc_src) and not os.path.exists("/content/seg_common.py"):
    shutil.copy(_sc_src, "/content/seg_common.py")
    print("Copied seg_common.py → /content/seg_common.py")

# Verify
print("\n/content/dataset/ layout:")
for root, dirs, files in os.walk(CONTENT_DATA):
    depth = root[len(CONTENT_DATA):].count(os.sep)
    indent = "  " * depth
    print(f"{indent}{os.path.basename(root) or root}/")
    for f in sorted(files)[:4]:
        print(f"{indent}  {f}")
    if len(files) > 4:
        print(f"{indent}  ... ({len(files)} files total)")

## 4 — Index all images and CSVs

In [ ]:
ALL_CSVS = sorted(glob.glob(os.path.join(CONTENT_DATA, "**", "*.csv"), recursive=True))
ALL_IMAGES = sorted(
    glob.glob(os.path.join(CONTENT_DATA, "**", "*.jpg"),  recursive=True) +
    glob.glob(os.path.join(CONTENT_DATA, "**", "*.jpeg"), recursive=True) +
    glob.glob(os.path.join(CONTENT_DATA, "**", "*.png"),  recursive=True)
)

print(f"CSVs  : {len(ALL_CSVS)}")
for p in ALL_CSVS:
    print(" ", p)
print(f"Images: {len(ALL_IMAGES)}  (first few below)")
for p in ALL_IMAGES[:4]:
    print(" ", p)

assert ALL_CSVS,   "No CSV files found under /content/dataset/"
assert ALL_IMAGES, "No image files found under /content/dataset/"

# filename -> full /content/ path
image_lookup = {os.path.basename(p): p for p in ALL_IMAGES}

# test images (used later for pred.csv)
TEST_IMAGE_PATHS = sorted(p for p in ALL_IMAGES if os.path.basename(p).startswith("test_"))
print(f"\nTrain images: {sum(1 for p in ALL_IMAGES if os.path.basename(p).startswith('train_'))}")
print(f"Test  images: {len(TEST_IMAGE_PATHS)}")

## 5 — Load all three annotation rounds and build majority-vote labels

Each of the three CSVs is an independent annotation of the same 5,000 training images.
We rasterise all three polygons per image and take the **pixel-wise majority vote** (foreground if ≥ 2/3 annotators agreed). Images with very low pairwise IoU (max < 0.3) are flagged as noisy and down-weighted during training.

In [ ]:
import pandas as pd, json as _json, numpy as _np

# ── Load all three CSVs ───────────────────────────────────────────────────────
csv_map = {}
for p in ALL_CSVS:
    name = os.path.basename(p)
    if name in ("train_round_1.csv", "train_round_2.csv", "train_round_3.csv"):
        df = pd.read_csv(p)
        assert "image" in df.columns and "polygon" in df.columns
        df = df.drop_duplicates(subset="image").set_index("image")
        csv_map[name] = df
        print(f"Loaded {name}: {len(df)} rows")

assert len(csv_map) == 3, f"Expected 3 round CSVs, found: {list(csv_map.keys())}"

# ── Build per-image records with majority-vote mask info ─────────────────────
# We'll compute the actual majority-vote mask in the Dataset __getitem__
# (to avoid storing large arrays). Here we pre-compute per-image agreement scores
# so we can log the distribution and optionally filter noisy samples.

import cv2 as _cv2
from tqdm.auto import tqdm as _tqdm

_THUMB = 128   # low-res thumbnail for fast IoU computation during agreement check

def _poly_to_mask_thumb(poly_json_str, size=_THUMB):
    """Rasterise a polygon JSON string to a boolean thumbnail mask."""
    try:
        polys = _json.loads(poly_json_str) if isinstance(poly_json_str, str) else poly_json_str
    except Exception:
        return _np.zeros((size, size), bool)
    mask = _np.zeros((size, size), _np.uint8)
    for poly in polys:
        if poly and len(poly) >= 3:
            pts = (_np.array(poly, _np.float32) * size).astype(_np.int32)
            _cv2.fillPoly(mask, [pts], 1)
    return mask.astype(bool)

def _iou_bool(a, b):
    inter = (_np.logical_and(a, b)).sum()
    union = (_np.logical_or(a, b)).sum()
    return float(inter) / float(union) if union > 0 else 0.0

# Gather all image names present in all three rounds
all_names = set(csv_map["train_round_1.csv"].index) \
          & set(csv_map["train_round_2.csv"].index) \
          & set(csv_map["train_round_3.csv"].index)
all_names = sorted(all_names)
print(f"Images present in all 3 rounds: {len(all_names)}")

# Build records list with per-image polygons from all 3 rounds
records = []
agreement_scores = []
noisy_count = 0

print("Computing annotator agreement scores...")
for name in _tqdm(all_names):
    if name not in image_lookup:
        continue
    polys_r1 = csv_map["train_round_1.csv"].loc[name, "polygon"]
    polys_r2 = csv_map["train_round_2.csv"].loc[name, "polygon"]
    polys_r3 = csv_map["train_round_3.csv"].loc[name, "polygon"]

    m1 = _poly_to_mask_thumb(polys_r1)
    m2 = _poly_to_mask_thumb(polys_r2)
    m3 = _poly_to_mask_thumb(polys_r3)

    iou_12 = _iou_bool(m1, m2)
    iou_13 = _iou_bool(m1, m3)
    iou_23 = _iou_bool(m2, m3)
    max_iou = max(iou_12, iou_13, iou_23)
    mean_iou = (iou_12 + iou_13 + iou_23) / 3.0
    agreement_scores.append(mean_iou)

    is_noisy = max_iou < 0.3
    if is_noisy:
        noisy_count += 1

    records.append({
        "image_name": name,
        "image_path": image_lookup[name],
        "polygons_r1": _json.loads(polys_r1) if isinstance(polys_r1, str) else polys_r1,
        "polygons_r2": _json.loads(polys_r2) if isinstance(polys_r2, str) else polys_r2,
        "polygons_r3": _json.loads(polys_r3) if isinstance(polys_r3, str) else polys_r3,
        "agreement":   mean_iou,
        "is_noisy":    is_noisy,
    })

import numpy as np
agreement_arr = np.array(agreement_scores)
print(f"\nAnnotator agreement (mean pairwise IoU) distribution:")
print(f"  mean={agreement_arr.mean():.3f}  median={np.median(agreement_arr):.3f}  "
      f"min={agreement_arr.min():.3f}  max={agreement_arr.max():.3f}")
print(f"  Noisy images (max pairwise IoU < 0.3): {noisy_count} / {len(records)} "
      f"({100*noisy_count/max(len(records),1):.1f}%)")
print(f"  Total records built: {len(records)}")

## 6 — Write `seg_common.py`

In [ ]:
%%writefile /content/seg_common.py
"""Shared U-Net code for the document segmentation task."""
import csv, glob, json, os, random
import cv2, numpy as np, torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

DATA_ROOT = "/content/dataset"
PRED_ROOT = "/content/results"
CKPT_ROOT = "/content/checkpoints"

IMG_SIZE = 256
IOU_THR  = 0.75
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)


# ──────────────────────────────────────────────────────────────────────────────
# Building blocks
# ──────────────────────────────────────────────────────────────────────────────

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)


# ──────────────────────────────────────────────────────────────────────────────
# Original U-Net  (DO NOT MODIFY)
# ──────────────────────────────────────────────────────────────────────────────

class UNet(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        self.d1 = DoubleConv(3, base);        self.d2 = DoubleConv(base, base*2)
        self.d3 = DoubleConv(base*2, base*4); self.d4 = DoubleConv(base*4, base*8)
        self.pool = nn.MaxPool2d(2)
        self.up3 = nn.ConvTranspose2d(base*8, base*4, 2, stride=2); self.u3 = DoubleConv(base*8, base*4)
        self.up2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2); self.u2 = DoubleConv(base*4, base*2)
        self.up1 = nn.ConvTranspose2d(base*2, base,   2, stride=2); self.u1 = DoubleConv(base*2, base)
        self.out = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        e1 = self.d1(x); e2 = self.d2(self.pool(e1))
        e3 = self.d3(self.pool(e2)); bn = self.d4(self.pool(e3))
        d3 = self.u3(torch.cat([self.up3(bn), e3], 1))
        d2 = self.u2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.u1(torch.cat([self.up1(d2), e1], 1))
        return self.out(d1)


# ──────────────────────────────────────────────────────────────────────────────
# Boundary-Aware U-Net extension
# ──────────────────────────────────────────────────────────────────────────────

class BoundaryAwareUNet(nn.Module):
    """Extends UNet with a lightweight edge decoder head.

    forward(x) returns a tuple (seg_logits, edge_logits), both [B, 1, H, W].
    The seg_head reuses the original UNet decoder exactly.
    The edge_head is a small decoder that takes e1 and e2 skip features.
    """
    def __init__(self, base=32):
        super().__init__()
        # ── Encoder (identical to UNet) ──────────────────────────────────────
        self.d1   = DoubleConv(3, base)
        self.d2   = DoubleConv(base, base * 2)
        self.d3   = DoubleConv(base * 2, base * 4)
        self.d4   = DoubleConv(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)

        # ── Segmentation decoder (identical to UNet seg_head) ────────────────
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.u3  = DoubleConv(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.u2  = DoubleConv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.u1  = DoubleConv(base * 2, base)
        self.out = nn.Conv2d(base, 1, 1)

        # ── Edge decoder head (lightweight) ─────────────────────────────────
        # Takes e2 (base*2 channels, at 1/2 resolution) and upsample to
        # full resolution using e1 skip, then predict edge logits.
        self.edge_up2  = nn.ConvTranspose2d(base * 2, base, 2, stride=2)  # 1/2 -> full
        self.edge_fuse = DoubleConv(base * 2, base)   # cat(edge_up2, e1) -> base
        self.edge_out  = nn.Conv2d(base, 1, 1)

    def forward(self, x):
        # Encoder
        e1 = self.d1(x)                       # full res,  base ch
        e2 = self.d2(self.pool(e1))            # 1/2 res,   base*2 ch
        e3 = self.d3(self.pool(e2))            # 1/4 res,   base*4 ch
        bn = self.d4(self.pool(e3))            # 1/8 res,   base*8 ch

        # Segmentation decoder
        d3 = self.u3(torch.cat([self.up3(bn), e3], 1))
        d2 = self.u2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.u1(torch.cat([self.up1(d2), e1], 1))
        seg_logits = self.out(d1)

        # Edge decoder
        eu = self.edge_up2(e2)                 # upsample e2 to full res
        ef = self.edge_fuse(torch.cat([eu, e1], 1))  # fuse with e1
        edge_logits = self.edge_out(ef)

        return seg_logits, edge_logits


# ──────────────────────────────────────────────────────────────────────────────
# Loss functions
# ──────────────────────────────────────────────────────────────────────────────

def dice_loss(logits, target, eps=1e-6):
    prob = torch.sigmoid(logits)
    num  = 2 * (prob * target).sum(dim=(1,2,3)) + eps
    den  = prob.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3)) + eps
    return (1 - num / den).mean()


def combined_loss(seg_logit, edge_logit, seg_target, edge_target):
    """Segmentation BCE+Dice loss combined with edge BCE loss.

    seg_loss  = BCEWithLogits(seg_logit, seg_target) + dice_loss(seg_logit, seg_target)
    edge_loss = BCEWithLogits(edge_logit, edge_target)
    total     = seg_loss + 0.4 * edge_loss
    """
    bce_fn  = torch.nn.BCEWithLogitsLoss()
    seg_bce = bce_fn(seg_logit, seg_target)
    seg_dic = dice_loss(seg_logit, seg_target)
    seg_loss  = seg_bce + seg_dic
    edge_loss = bce_fn(edge_logit, edge_target)
    return seg_loss + 0.4 * edge_loss, seg_loss.item(), edge_loss.item()


# ──────────────────────────────────────────────────────────────────────────────
# Edge target computation
# ──────────────────────────────────────────────────────────────────────────────

def make_edge_target(mask_tensor):
    """Compute edge map from a binary mask tensor [B, 1, H, W] float {0,1}.

    Uses a 3x3 Sobel kernel via F.conv2d. Returns a [B, 1, H, W] float
    tensor with values in [0, 1] (1 = edge pixel).
    """
    # Sobel kernels
    kx = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
                      dtype=mask_tensor.dtype, device=mask_tensor.device)
    ky = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
                      dtype=mask_tensor.dtype, device=mask_tensor.device)
    # Shape: [out_ch, in_ch, H, W]
    kx = kx.view(1, 1, 3, 3)
    ky = ky.view(1, 1, 3, 3)

    gx = F.conv2d(mask_tensor, kx, padding=1)
    gy = F.conv2d(mask_tensor, ky, padding=1)
    edge = torch.sqrt(gx ** 2 + gy ** 2 + 1e-8)
    # Normalise: clamp to [0, 1]
    edge = edge / (edge.max().clamp(min=1e-8))
    return (edge > 0.1).float()


# ──────────────────────────────────────────────────────────────────────────────
# Data helpers
# ──────────────────────────────────────────────────────────────────────────────

def polys_to_mask(polygons, H, W):
    mask = np.zeros((H, W), np.uint8)
    for poly in polygons:
        if poly and len(poly) >= 3:
            pts = (np.array(poly, np.float32) * np.array([W, H])).astype(np.int32)
            cv2.fillPoly(mask, [pts], 255)
    return mask


class CsvSegDataset(Dataset):
    """Records: list of {image_name, image_path, polygons_r1, polygons_r2, polygons_r3}.

    __getitem__ returns (image_tensor, seg_mask, edge_mask) triples.
    The seg_mask is the majority-vote of the three polygon annotations.
    """
    def __init__(self, records, transform):
        self.records = records
        self.transform = transform

    def __len__(self): return len(self.records)

    def __getitem__(self, i):
        r   = self.records[i]
        img = cv2.cvtColor(cv2.imread(r["image_path"]), cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]

        # Rasterise all three annotations
        m1 = polys_to_mask(r["polygons_r1"], H, W).astype(np.uint8)
        m2 = polys_to_mask(r["polygons_r2"], H, W).astype(np.uint8)
        m3 = polys_to_mask(r["polygons_r3"], H, W).astype(np.uint8)

        # Majority vote: pixel is foreground if >= 2 of 3 annotators marked it
        vote = ((m1 > 127).astype(np.uint8) +
                (m2 > 127).astype(np.uint8) +
                (m3 > 127).astype(np.uint8))
        mask = (vote >= 2).astype(np.uint8) * 255

        t = self.transform(image=img, mask=mask)
        seg_mask = (t["mask"] > 127).float().unsqueeze(0)  # [1, H, W]

        # Edge target from the segmentation mask
        edge_mask = make_edge_target(seg_mask.unsqueeze(0)).squeeze(0)  # [1, H, W]

        return t["image"], seg_mask, edge_mask


# ──────────────────────────────────────────────────────────────────────────────
# Metrics
# ──────────────────────────────────────────────────────────────────────────────

def dice_coeff(pred, gt, eps=1e-6):
    pred = np.asarray(pred).astype(bool); gt = np.asarray(gt).astype(bool)
    return float((2 * np.logical_and(pred, gt).sum() + eps) / (pred.sum() + gt.sum() + eps))


# ──────────────────────────────────────────────────────────────────────────────
# Inference helpers
# ──────────────────────────────────────────────────────────────────────────────

def preprocess(img_rgb):
    r = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    n = (r.astype(np.float32) / 255.0 - MEAN) / STD
    return torch.from_numpy(n.transpose(2, 0, 1)).unsqueeze(0).float()


def fused_inference(model, image_rgb, thresh=0.5, alpha=0.3):
    """Run BoundaryAwareUNet inference and fuse seg + edge predictions.

    prob = sigmoid(seg_logits) * (1 + alpha * sigmoid(edge_logits))
    Returns a binary numpy uint8 mask at IMG_SIZE resolution.
    """
    model.eval()
    with torch.no_grad():
        x = preprocess(image_rgb).to(DEVICE)
        seg_logits, edge_logits = model(x)
        seg_prob  = torch.sigmoid(seg_logits)
        edge_prob = torch.sigmoid(edge_logits)
        prob = seg_prob * (1.0 + alpha * edge_prob)
        prob = prob.clamp(0.0, 1.0)
        binary = (prob[0, 0].cpu().numpy() > thresh).astype(np.uint8)
    return binary


# ──────────────────────────────────────────────────────────────────────────────
# Polygon post-processing
# ──────────────────────────────────────────────────────────────────────────────

def mask_to_polygons(binary, size=None, min_area=80, epsilon_frac=0.01):
    """Convert a binary mask to a list of normalised polygons.

    Applies morphological cleaning before contouring, and uses epsilon_frac=0.01
    for slightly more aggressive simplification to reduce jagged pixel noise.
    """
    if size is None: size = IMG_SIZE
    binary = binary.astype(np.uint8)

    # Morphological cleaning
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=2)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN,  kernel, iterations=1)

    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polygons = []
    for c in contours:
        if cv2.contourArea(c) < min_area: continue
        approx = cv2.approxPolyDP(c, epsilon_frac * cv2.arcLength(c, True), True)
        if len(approx) < 3: continue
        pts = approx.reshape(-1, 2).astype(np.float32) / size
        polygons.append([[round(float(x),5), round(float(y),5)] for x,y in pts])
    return polygons

## 7 — Imports and config

In [ ]:
import sys, random, csv, json
import numpy as np, cv2, torch, torch.nn as nn
from torch.utils.data import DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

sys.path.insert(0, "/content")
import importlib, seg_common as sc
importlib.reload(sc)   # ensure the %%writefile version is active

os.makedirs(sc.CKPT_ROOT, exist_ok=True)
os.makedirs(sc.PRED_ROOT, exist_ok=True)

torch.manual_seed(0); np.random.seed(0); random.seed(0)
print("device:", sc.DEVICE, "| torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

## 8 — Training hyper-parameters

In [ ]:
MODEL_NAME  = "boundary_unet"
BATCH       = 16
EPOCHS      = 30
LR          = 3e-4
VAL_FRAC    = 0.10
NUM_WORKERS = 0   # 0 avoids multiprocessing issues on Python 3.12 / Colab

BEST_CKPT_PATH = os.path.join(sc.CKPT_ROOT, f"{MODEL_NAME}_best.pt")
LAST_CKPT_PATH = os.path.join(sc.CKPT_ROOT, f"{MODEL_NAME}_last.pt")

print(f"model: {MODEL_NAME}  |  img_size {sc.IMG_SIZE}  |  batch {BATCH}  |  epochs {EPOCHS}  |  lr {LR}")
print(f"best ckpt → {BEST_CKPT_PATH}")
print(f"last ckpt → {LAST_CKPT_PATH}")

## 9 — Transforms

In [ ]:
train_tf = A.Compose([
    A.Resize(sc.IMG_SIZE, sc.IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomRotate90(p=0.5),
    A.Perspective(scale=(0.02, 0.08), p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=sc.MEAN, std=sc.STD),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.Resize(sc.IMG_SIZE, sc.IMG_SIZE),
    A.Normalize(mean=sc.MEAN, std=sc.STD),
    ToTensorV2(),
])

## 10 — Build train / val split

In [ ]:
import random as _rnd
_rnd.Random(0).shuffle(records)

n_val         = int(len(records) * VAL_FRAC)
val_records   = records[:n_val]
train_records = records[n_val:]

train_ds = sc.CsvSegDataset(train_records, train_tf)
val_ds   = sc.CsvSegDataset(val_records,   val_tf)

train_ld = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_ld   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)

print(f"train: {len(train_ds)} images  |  val: {len(val_ds)} images")
print(f"train batches per epoch: {len(train_ld)}")

## 11 — Model

In [ ]:
model = sc.BoundaryAwareUNet(base=32).to(sc.DEVICE)
opt   = torch.optim.Adam(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)

total_params = sum(p.numel() for p in model.parameters())
print(f"{total_params/1e6:.2f}M params  |  device: {sc.DEVICE}")

## 12 — Train

In [ ]:
@torch.no_grad()
def eval_dice(loader):
    model.eval()
    scores = []
    for x, seg_y, _edge_y in loader:
        x = x.to(sc.DEVICE)
        seg_logit, _ = model(x)   # use only seg head for validation
        prob = torch.sigmoid(seg_logit)
        for p, g in zip(prob.cpu().numpy(), seg_y.numpy()):
            scores.append(sc.dice_coeff(p > 0.5, g))
    return float(np.mean(scores))


hist = {"train_loss": [], "seg_loss": [], "edge_loss": [], "val_dice": []}
best_val_dice = 0.0

for ep in range(1, EPOCHS + 1):
    model.train()
    running_total = 0.0
    running_seg   = 0.0
    running_edge  = 0.0
    n = 0

    pbar = tqdm(train_ld, desc=f"epoch {ep}/{EPOCHS}", leave=False)
    for x, seg_y, edge_y in pbar:
        x, seg_y, edge_y = x.to(sc.DEVICE), seg_y.to(sc.DEVICE), edge_y.to(sc.DEVICE)
        opt.zero_grad()
        seg_logit, edge_logit = model(x)
        loss, seg_l, edge_l = sc.combined_loss(seg_logit, edge_logit, seg_y, edge_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()

        running_total += loss.item()
        running_seg   += seg_l
        running_edge  += edge_l
        n += 1
        pbar.set_postfix(loss=running_total / n)

    sched.step()
    val_dice = eval_dice(val_ld)

    avg_total = running_total / n
    avg_seg   = running_seg   / n
    avg_edge  = running_edge  / n

    hist["train_loss"].append(avg_total)
    hist["seg_loss"].append(avg_seg)
    hist["edge_loss"].append(avg_edge)
    hist["val_dice"].append(val_dice)

    print(f"epoch {ep:2d}  loss {avg_total:.4f}  "
          f"seg_loss {avg_seg:.4f}  edge_loss {avg_edge:.4f}  "
          f"val_dice {val_dice:.4f}")

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(model.state_dict(), BEST_CKPT_PATH)
        print(f"  ↑ new best val_dice: {val_dice:.4f}  → saved {BEST_CKPT_PATH}")

# Save the final epoch as the last checkpoint
torch.save(model.state_dict(), LAST_CKPT_PATH)
print(f"\nLast checkpoint saved → {LAST_CKPT_PATH}")
print(f"Best val_dice: {best_val_dice:.4f}  → {BEST_CKPT_PATH}")

## 13 — Training curves

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(hist["train_loss"], marker="o", label="total")
ax[0].plot(hist["seg_loss"],   marker="s", label="seg")
ax[0].plot(hist["edge_loss"],  marker="^", label="edge")
ax[0].set_title("train loss"); ax[0].set_xlabel("epoch"); ax[0].legend()

ax[1].plot(hist["val_dice"],   marker="o", color="green")
ax[1].set_title("val Dice");   ax[1].set_xlabel("epoch"); ax[1].set_ylim(0, 1)

ax[2].plot(hist["seg_loss"],   marker="s", color="orange", label="seg")
ax[2].plot(hist["edge_loss"],  marker="^", color="red",    label="edge")
ax[2].set_title("seg vs edge loss"); ax[2].set_xlabel("epoch"); ax[2].legend()

plt.tight_layout(); plt.show()

## 14 — Checkpoint management

The best checkpoint (by val_dice) is saved automatically during training as `BEST_CKPT_PATH`.
The final epoch checkpoint is saved as `LAST_CKPT_PATH`.
Inference (Section 15) loads the **best** checkpoint.

In [ ]:
print(f"Best checkpoint : {BEST_CKPT_PATH}  (val_dice={best_val_dice:.4f})")
print(f"Last checkpoint : {LAST_CKPT_PATH}")
print(f"Best file exists: {os.path.exists(BEST_CKPT_PATH)}")
print(f"Last file exists: {os.path.exists(LAST_CKPT_PATH)}")

---
## 15 — Generate `pred.csv`

You can run this section **independently** after a session reset — just re-run Sections 1–7 first.

Loads the **best** checkpoint by default. Override `CKPT_TO_USE` if you want the last epoch instead.

In [ ]:
# ── Set which checkpoint to use (default: best) ──────────────────────────────
CKPT_TO_USE = BEST_CKPT_PATH   # or LAST_CKPT_PATH for the final epoch
# ─────────────────────────────────────────────────────────────────────────────

model = sc.BoundaryAwareUNet(base=32).to(sc.DEVICE)
model.load_state_dict(torch.load(CKPT_TO_USE, map_location=sc.DEVICE))
model.eval()
print(f"Loaded: {CKPT_TO_USE}  |  device: {sc.DEVICE}")

In [ ]:
if not TEST_IMAGE_PATHS:
    print("No test images found in /content/dataset/. Check the dataset structure.")
else:
    print(f"Running inference on {len(TEST_IMAGE_PATHS)} test images...")
    rows = []
    for img_path in tqdm(TEST_IMAGE_PATHS, desc="predicting"):
        image  = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        binary = sc.fused_inference(model, image, thresh=0.5, alpha=0.3)
        polygons = sc.mask_to_polygons(binary)
        rows.append((os.path.basename(img_path), json.dumps(polygons)))

    PRED_CSV = os.path.join(sc.PRED_ROOT, "pred.csv")
    with open(PRED_CSV, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["image", "polygon"])
        w.writerows(rows)
    print(f"Wrote {len(rows)} rows → {PRED_CSV}")

## 16 — Submit predictions

In [ ]:
import requests

# ── CONFIG: edit these ────────────────────────────────────────────────────────

BASE_URL        = "http://3.108.8.61:8991"
HIRING_CODE     = "CIT_DE_2026"

PREDICTIONS_CSV = "predictions.csv"        # path to YOUR predictions
NAME            = "Vaishnavi U"
EMAIL           = "71762232058@cit.edu.in"     # your unique ID -- use the same one every time

# ──────────────────────────────────────────────────────────────────────────────


def submit_predictions(pred_path: str, name: str, email: str, hiring_code: str) -> dict:
    url = f"{BASE_URL}/submit"
    with open(pred_path, "rb") as f:
        files = {"predictions": (pred_path, f, "text/csv")}
        data = {"name": name, "email": email, "hiring_code": hiring_code}
        resp = requests.post(url, data=data, files=files, timeout=300)

    if not resp.ok:
        print(f"Request failed: {resp.status_code}")
        try:
            print(resp.json().get("detail", resp.text))
        except ValueError:
            print(resp.text)
        return {}

    return resp.json()


def print_results(result: dict) -> None:
    if not result:
        return

    s = result["this_submission"]
    b = result["your_best"]

    print()
    print(f'Images evaluated   : {s["n_images"]}')
    print(f'Pixel Dice (mean)  : {s["dice_mean"]:.4f}')
    print(f'Instance Precision : {s["precision"]:.4f}  (TP={s["tp"]}  FP={s["fp"]})')
    print(f'Instance Recall    : {s["recall"]:.4f}  (FN={s["fn"]})')
    print(f'Instance F1        : {s["f1"]:.4f}  (IoU thr = {s["iou_thr"]:.2f})')
    print()
    print(f'Best Dice so far   : {b["dice_mean"]:.4f}  (attempt {b["attempt"]})')
    print(f'Attempts so far    : {result["attempt"]}')
    print(f'Attempts remaining : {result["attempts_remaining"]}')


print_results(submit_predictions(
    PREDICTIONS_CSV,
    NAME,
    EMAIL,
    HIRING_CODE,
))

In [ ]:
from google.colab import files
files.download(PRED_CSV)